# Notebook 46: Feedback Loop

This notebook demonstrates reward and feedback management using `multigen.feedback_loop`.
All examples use mock implementations — no real API keys are required.

Topics covered:
- `OutcomeStore` — add outcomes, query by agent/workflow, mean value
- `DelayedRewardBuffer` — register runs, resolve with delay, expire old entries
- `RewardShaper` — clip / zscore / rank normalisation modes
- `HumanFeedbackStore` — thumbs-up/down, mean score, flagged entries
- `FeedbackLoopManager` facade — end-to-end ingest, defer, resolve, human feedback, aggregate

In [ ]:
import sys
sys.path.insert(0, '../sdk')


## Mock Implementation of `multigen.feedback_loop`

In [ ]:
import statistics
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

# ---------------------------------------------------------------------------
# 1. OutcomeStore
# ---------------------------------------------------------------------------

@dataclass
class Outcome:
    run_id: str
    agent_id: str
    workflow_id: str
    value: float
    metadata: Dict = field(default_factory=dict)


class OutcomeStore:
    def __init__(self):
        self._outcomes: List[Outcome] = []

    def add(self, run_id: str, agent_id: str, workflow_id: str, value: float, **meta):
        self._outcomes.append(Outcome(
            run_id=run_id, agent_id=agent_id,
            workflow_id=workflow_id, value=value, metadata=meta
        ))

    def query(self, agent_id: str = None, workflow_id: str = None) -> List[Outcome]:
        result = self._outcomes
        if agent_id:
            result = [o for o in result if o.agent_id == agent_id]
        if workflow_id:
            result = [o for o in result if o.workflow_id == workflow_id]
        return result

    def mean_value(self, agent_id: str = None, workflow_id: str = None) -> float:
        outcomes = self.query(agent_id=agent_id, workflow_id=workflow_id)
        if not outcomes:
            return 0.0
        return round(statistics.mean(o.value for o in outcomes), 4)

print('OutcomeStore defined.')


## Section 1 — OutcomeStore

In [ ]:
store = OutcomeStore()
store.add('run_001', agent_id='agent_A', workflow_id='wf_search', value=0.85)
store.add('run_002', agent_id='agent_A', workflow_id='wf_search', value=0.72)
store.add('run_003', agent_id='agent_B', workflow_id='wf_search', value=0.60)
store.add('run_004', agent_id='agent_A', workflow_id='wf_report', value=0.90)
store.add('run_005', agent_id='agent_B', workflow_id='wf_report', value=0.55)

print('All outcomes:')
for o in store.query():
    print(f'  {o.run_id}: agent={o.agent_id}  workflow={o.workflow_id}  value={o.value}')

print(f'\nmean_value(agent_A)            = {store.mean_value(agent_id="agent_A")}')
print(f'mean_value(agent_B)            = {store.mean_value(agent_id="agent_B")}')
print(f'mean_value(wf_search)          = {store.mean_value(workflow_id="wf_search")}')
print(f'mean_value(agent_A, wf_search) = {store.mean_value(agent_id="agent_A", workflow_id="wf_search")}')


## Section 2 — DelayedRewardBuffer

In [ ]:
@dataclass
class PendingRun:
    run_id: str
    agent_id: str
    registered_at: float = field(default_factory=time.time)
    resolved: bool = False
    reward: Optional[float] = None


class DelayedRewardBuffer:
    def __init__(self, ttl_seconds: float = 3600):
        self.ttl = ttl_seconds
        self._pending: Dict[str, PendingRun] = {}

    def register(self, run_id: str, agent_id: str):
        self._pending[run_id] = PendingRun(run_id=run_id, agent_id=agent_id)
        print(f'  Registered {run_id} (agent={agent_id})')

    def resolve(self, run_id: str, reward: float):
        if run_id not in self._pending:
            raise KeyError(f'Unknown run_id: {run_id}')
        entry = self._pending[run_id]
        entry.resolved = True
        entry.reward = reward
        print(f'  Resolved {run_id}: reward={reward}')

    def expire_old(self, now: float = None) -> List[str]:
        now = now or time.time()
        expired = [
            rid for rid, r in self._pending.items()
            if not r.resolved and (now - r.registered_at) > self.ttl
        ]
        for rid in expired:
            del self._pending[rid]
        return expired

    def resolved_rewards(self) -> List[PendingRun]:
        return [r for r in self._pending.values() if r.resolved]


buf = DelayedRewardBuffer(ttl_seconds=60)

print('Registering runs:')
for rid in ['run_010', 'run_011', 'run_012', 'run_013']:
    buf.register(rid, agent_id='agent_C')

# Simulate time passing for run_010 and run_011 (manually set timestamp)
buf._pending['run_010'].registered_at = time.time() - 120  # 2 min ago → expired
buf._pending['run_011'].registered_at = time.time() - 90   # 90s ago → expired

print('\nResolving run_012 and run_013:')
buf.resolve('run_012', reward=0.88)
buf.resolve('run_013', reward=0.45)

expired = buf.expire_old()
print(f'\nExpired (unresolved, past TTL): {expired}')

print(f'\nResolved rewards:')
for r in buf.resolved_rewards():
    print(f'  {r.run_id}: agent={r.agent_id}  reward={r.reward}')


## Section 3 — RewardShaper

In [ ]:
class RewardShaper:
    """Normalise a stream of raw reward values using clip, zscore, or rank modes."""

    @staticmethod
    def clip(values: List[float], lo: float = 0.0, hi: float = 1.0) -> List[float]:
        return [round(max(lo, min(hi, v)), 4) for v in values]

    @staticmethod
    def zscore(values: List[float]) -> List[float]:
        if len(values) < 2:
            return [0.0] * len(values)
        mu = statistics.mean(values)
        sigma = statistics.stdev(values) or 1e-9
        return [round((v - mu) / sigma, 4) for v in values]

    @staticmethod
    def rank(values: List[float]) -> List[float]:
        n = len(values)
        if n == 0:
            return []
        sorted_vals = sorted(enumerate(values), key=lambda x: x[1])
        result = [0.0] * n
        for rank_idx, (orig_idx, _) in enumerate(sorted_vals):
            result[orig_idx] = round(rank_idx / (n - 1), 4) if n > 1 else 1.0
        return result


raw = [-0.5, 0.1, 0.3, 0.7, 1.2, 2.0, 0.55, -0.1]
print(f'Raw values : {raw}')
print(f'clip [0,1] : {RewardShaper.clip(raw)}')
print(f'zscore     : {RewardShaper.zscore(raw)}')
print(f'rank [0,1] : {RewardShaper.rank(raw)}')


## Section 4 — HumanFeedbackStore

In [ ]:
@dataclass
class HumanFeedback:
    run_id: str
    rating: int   # +1 (thumbs up) or -1 (thumbs down)
    comment: str = ''
    flagged: bool = False


class HumanFeedbackStore:
    def __init__(self):
        self._records: List[HumanFeedback] = []

    def record(self, run_id: str, thumbs_up: bool, comment: str = '', flag: bool = False):
        rating = 1 if thumbs_up else -1
        self._records.append(HumanFeedback(run_id=run_id, rating=rating,
                                            comment=comment, flagged=flag))

    def mean_score(self, run_id: str = None) -> float:
        records = self._records
        if run_id:
            records = [r for r in records if r.run_id == run_id]
        if not records:
            return 0.0
        return round(statistics.mean(r.rating for r in records), 4)

    def flagged(self) -> List[HumanFeedback]:
        return [r for r in self._records if r.flagged]


hf = HumanFeedbackStore()
hf.record('run_020', thumbs_up=True,  comment='Great response!')
hf.record('run_021', thumbs_up=False, comment='Missed the point', flag=True)
hf.record('run_022', thumbs_up=True)
hf.record('run_023', thumbs_up=False, comment='Hallucinated a fact', flag=True)
hf.record('run_024', thumbs_up=True,  comment='Very helpful')

print(f'Overall mean_score : {hf.mean_score()}')
print(f'Mean for run_020   : {hf.mean_score("run_020")}')
print(f'Mean for run_021   : {hf.mean_score("run_021")}')

flagged = hf.flagged()
print(f'\nFlagged entries ({len(flagged)}):')
for f in flagged:
    print(f'  {f.run_id}: rating={f.rating:+d}  comment={f.comment!r}')


## Section 5 — FeedbackLoopManager Facade

In [ ]:
class FeedbackLoopManager:
    """
    Facade combining OutcomeStore, DelayedRewardBuffer,
    RewardShaper, and HumanFeedbackStore.
    """

    def __init__(self, reward_ttl: float = 3600):
        self.outcomes = OutcomeStore()
        self.deferred = DelayedRewardBuffer(ttl_seconds=reward_ttl)
        self.human_fb = HumanFeedbackStore()

    def ingest(self, run_id: str, agent_id: str, workflow_id: str, immediate_value: float):
        self.outcomes.add(run_id, agent_id=agent_id,
                          workflow_id=workflow_id, value=immediate_value)
        print(f'  [ingest] {run_id}: value={immediate_value}')

    def defer(self, run_id: str, agent_id: str):
        self.deferred.register(run_id, agent_id)

    def resolve(self, run_id: str, reward: float):
        self.deferred.resolve(run_id, reward)

    def record_human(self, run_id: str, thumbs_up: bool, comment: str = '', flag: bool = False):
        self.human_fb.record(run_id, thumbs_up, comment, flag)

    def aggregate_for_agent(self, agent_id: str) -> Dict:
        outcome_mean = self.outcomes.mean_value(agent_id=agent_id)
        deferred_rewards = [
            r.reward for r in self.deferred.resolved_rewards()
            if r.agent_id == agent_id and r.reward is not None
        ]
        deferred_mean = round(statistics.mean(deferred_rewards), 4) if deferred_rewards else None
        human_mean = self.human_fb.mean_score()
        return {
            'agent_id': agent_id,
            'outcome_mean': outcome_mean,
            'deferred_mean': deferred_mean,
            'human_mean': human_mean,
            'flagged_count': len(self.human_fb.flagged()),
        }


mgr = FeedbackLoopManager(reward_ttl=30)

print('Ingesting immediate outcomes:')
mgr.ingest('run_100', 'agent_X', 'wf_pipeline', 0.80)
mgr.ingest('run_101', 'agent_X', 'wf_pipeline', 0.75)
mgr.ingest('run_102', 'agent_Y', 'wf_pipeline', 0.60)

print('\nDeferring rewards:')
mgr.defer('run_100', 'agent_X')
mgr.defer('run_101', 'agent_X')

print('\nResolving deferred rewards:')
mgr.resolve('run_100', reward=0.92)
mgr.resolve('run_101', reward=0.78)

print('\nRecording human feedback:')
mgr.record_human('run_100', thumbs_up=True,  comment='Spot on!')
mgr.record_human('run_101', thumbs_up=True)
mgr.record_human('run_102', thumbs_up=False, comment='Wrong answer', flag=True)

print('\nAggregate for agent_X:')
agg = mgr.aggregate_for_agent('agent_X')
for k, v in agg.items():
    print(f'  {k}: {v}')
